In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [4]:
train_data = torchvision.datasets.FashionMNIST(root='./data',
                              train=True,
                              download=True,
                              transform=transform)

test_data = torchvision.datasets.FashionMNIST(root='./data',
                              train=False,
                              download=True,
                              transform=transform)

In [5]:
train_loader=DataLoader(train_data, batch_size=64, shuffle=True)
test_loader=DataLoader(test_data, batch_size=64, shuffle=False)

In [6]:
# 신경망
class Con(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(1,6,5,padding=2)
        self.conv2=nn.Conv2d(6,16,5)

        self.fc1=nn.Linear(16*5*5, 120)
        self.fc2=nn.Linear(120, 84)
        self.fc3=nn.Linear(84, 10)

        self.maxpool=nn.MaxPool2d(2)
        self.relu=nn.ReLU()

    def forward(self, x):
        x=self.relu(self.conv1(x))
        x=self.maxpool(x)

        
        x=self.relu(self.conv2(x))
        x=self.maxpool(x)

        x=torch.flatten(x,1)
        x=self.relu(self.fc1(x))
        x=self.relu(self.fc2(x))
        out=self.fc3(x)
        return out

In [7]:
model=Con()
crossloss=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(), lr=0.001)

In [8]:
epochs=10

loss1, acc1 =[], []

for epoch in range(epochs):
    model.train()
    t_loss, t_acc=0, 0
    
    for images, labels in train_loader:
        outputs=model(images)
        loss=crossloss(outputs, labels)
        
        optimizer.zero_grad() # 기울기 초기화
        loss.backward() # 기울기 계산
        optimizer.step() # 가중치 업데이트

        t_loss+=loss.item()
        _,predicted=torch.max(outputs,1)
        t_acc+=(predicted==labels).sum().item()
        
        
    epoch_loss=t_loss/len(train_loader.dataset)
    epoch_acc=t_acc/len(train_loader.dataset)
    
    loss1.append(epoch_loss)
    acc1.append(epoch_acc)

    print(t_loss)
    print(t_acc)

print(loss1)
print(acc1)

535.8170744478703
47566
340.4390708953142
52073
295.78586583584547
53012
267.916137970984
53653
247.2886217609048
54101
227.6680754944682
54586
215.64027129113674
54862
200.86803621798754
55250
189.0837085917592
55459
179.82251181825995
55671
[0.008930284574131172, 0.005673984514921904, 0.004929764430597425, 0.004465268966183066, 0.004121477029348414, 0.0037944679249078037, 0.0035940045215189457, 0.0033478006036331254, 0.0031513951431959867, 0.002997041863637666]
[0.7927666666666666, 0.8678833333333333, 0.8835333333333333, 0.8942166666666667, 0.9016833333333333, 0.9097666666666666, 0.9143666666666667, 0.9208333333333333, 0.9243166666666667, 0.92785]


In [9]:
import numpy as np

torch.save(model.state_dict(), 'fashion_mnist.pth')
np.save('loss1.npy', np.array(loss1))
np.save('acc1.npy', np.array(acc1))
print('저장함')

저장함
